In [ ]:
import pandas as pd
import requests

# ==========================================
# 1. CONFIGURATION
# ==========================================

YEAR = 2026

OUTPUT_FILE = "output/dim_calendar.csv"

# ==========================================
# 2. GENERATE CALENDAR
# ==========================================

dates = pd.date_range(
    start=f"{YEAR}-01-01",
    end=f"{YEAR}-12-31",
    freq="D"
)

calendar = pd.DataFrame({
    "attendance_date": dates
})

# ==========================================
# 3. GET INDONESIAN HOLIDAYS
# ==========================================

url = f"https://indonesian-holidays.ahrulsyamil.com/holidays/{YEAR}"

response = requests.get(url, timeout=30)
response.raise_for_status()

json_data = response.json()

holidays = pd.DataFrame(json_data["data"])

holidays["date"] = pd.to_datetime(holidays["date"])

# ==========================================
# 4. PREPARE HOLIDAY DATA
# ==========================================

holidays_grouped = (
    holidays.groupby("date")
    .agg(
        holiday_name=("name", lambda x: " | ".join(x)),
        holiday_type=("type", lambda x: " | ".join(x)),
        is_national_holiday=("is_national", "any")
    )
    .reset_index()
)

holidays_grouped = holidays_grouped.rename(
    columns={"date": "attendance_date"}
)

# ==========================================
# 5. MERGE CALENDAR + HOLIDAYS
# ==========================================

calendar = calendar.merge(
    holidays_grouped,
    on="attendance_date",
    how="left"
)

# ==========================================
# 6. CREATE CALENDAR ATTRIBUTES
# ==========================================

day_mapping = {
    0: "Senin",
    1: "Selasa",
    2: "Rabu",
    3: "Kamis",
    4: "Jumat",
    5: "Sabtu",
    6: "Minggu"
}

calendar["year"] = calendar["attendance_date"].dt.year
calendar["month"] = calendar["attendance_date"].dt.month
calendar["day"] = calendar["attendance_date"].dt.day
calendar["day_name"] = (
    calendar["attendance_date"].dt.dayofweek.map(day_mapping)
)

calendar["day_type"] = "Weekday"

calendar.loc[
    calendar["attendance_date"].dt.dayofweek >= 5,
    "day_type"
] = "Weekend"

calendar["is_weekend"] = (
    calendar["attendance_date"].dt.dayofweek >= 5
)

calendar["is_national_holiday"] = (
    calendar["is_national_holiday"].fillna(False)
)

calendar["is_joint_leave"] = (
    calendar["holiday_type"]
    .fillna("")
    .str.contains("joint_leave")
)

calendar["holiday_name"] = calendar["holiday_name"].fillna("")
calendar["holiday_type"] = calendar["holiday_type"].fillna("")

# ==========================================
# 7. EXPORT
# ==========================================

calendar.to_csv(
    OUTPUT_FILE,
    index=False,
    encoding="utf-8-sig"
)

print("Calendar successfully created!")
print("Total dates:", len(calendar))
print("National holidays:", calendar["is_national_holiday"].sum())
print("Joint leave:", calendar["is_joint_leave"].sum())

Calendar successfully created!
Total dates: 365
National holidays: 17
Joint leave: 8


C:\Users\PC IT - 1\AppData\Local\Temp\ipykernel_1324\1719543832.py:88: Pandas4Warning: Series.dt.dayofweek is deprecated and will be removed in a future version. Use Series.dt.day_of_week instead.
  calendar["attendance_date"].dt.dayofweek.map(day_mapping)
C:\Users\PC IT - 1\AppData\Local\Temp\ipykernel_1324\1719543832.py:94: Pandas4Warning: Series.dt.dayofweek is deprecated and will be removed in a future version. Use Series.dt.day_of_week instead.
  calendar["attendance_date"].dt.dayofweek >= 5,
C:\Users\PC IT - 1\AppData\Local\Temp\ipykernel_1324\1719543832.py:99: Pandas4Warning: Series.dt.dayofweek is deprecated and will be removed in a future version. Use Series.dt.day_of_week instead.
  calendar["attendance_date"].dt.dayofweek >= 5
